# Proposta 1 — Treinamento real dos oito classificadores na Area1

Este é o **próximo passo científico** do mestrado. Utiliza os **510 pontos originais** (257 árvores e 253 gramíneas) para reconstruir **505 segmentos rotulados**, selecionar regularização **somente na Area1** e treinar os oito modelos finais para aplicação às outras regiões.

A seguir, executa uma **nova validação espacial aninhada**: blocos de 250 m e 500 m, cinco divisões externas por escala, três divisões internas por partição externa e exclusão geométrica de 25 m.

**Segurança:** os TIFFs originais e GeoPackages permanecem no seu Drive. Nenhum raster, rótulo individual ou modelo é publicado automaticamente no repositório. O README principal e os números históricos não são alterados.

**Atenção:** o F1 obtido aqui mede validação *interna na Area1*. Não é F1 de generalização para Sagrada Família, Centro, Lomba Grande ou Cristo Rei.

## 1. Abrir Drive e obter a versão atual do código

Execute as células na ordem. Se você já montou o Drive em outra sessão do Colab, pode ser necessário autorizá-lo novamente nesta sessão.

In [ ]:
from pathlib import Path
import subprocess
import sys
from google.colab import drive

drive.mount('/content/drive')
REPO=Path('/content/mestrado-arvores-urbanas-externo')
if REPO.exists():
    subprocess.run(['git','-C',str(REPO),'fetch','origin','feature/validacao-geografica-externa'],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','feature/validacao-geografica-externa'],check=True)
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin','feature/validacao-geografica-externa'],check=True)
else:
    subprocess.run(['git','clone','--depth=1','--branch','feature/validacao-geografica-externa',
                    'https://github.com/BrenoBragaGalvao/mestrado-arvores-urbanas.git',
                    str(REPO)],check=True)

SCRIPTS=REPO/'experimentos/validacao_externa_regioes'
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(SCRIPTS/'requirements.txt')],check=True)
print('Código atualizado:',SCRIPTS)

## 2. Localizar a Area1 original e a pasta de resultados

Não selecionamos automaticamente dados de outra cidade. Os arquivos de referência precisam ser os **originais da Area1**.

In [ ]:
ROOT=Path('/content/drive/MyDrive')
CANDIDATOS=[
    ROOT/'Projetos/Segmentacao_Breno/Area1',
    ROOT/'Segmentacao_Breno/Area1',
    ROOT/'Area1',
]
AREA1=next((p for p in CANDIDATOS if
            (p/'AOI_RGB_NEW.tif').is_file() and
            (p/'AOI_NEW_MULTI.tif').is_file() and
            (p/'Arvores.gpkg').is_file() and
            (p/'Grama.gpkg').is_file()),None)
if AREA1 is None:
    raise FileNotFoundError(
        'Area1 não localizada nos caminhos usuais de Meu Drive. '
        'Confira se a pasta Projetos/Segmentacao_Breno/Area1 está acessível.')
SAIDA=ROOT/'Projetos/Segmentacao_Breno/outputs/validacao_externa_regioes/area1_treinamento_real'
SAIDA.mkdir(parents=True,exist_ok=True)
print('Entradas originais privadas:',AREA1)
print('Saídas novas:',SAIDA)

## 3. Auditar os 510 pontos e os rasters antes de treinar

Se houver qualquer divergência nos dados originais, o processo **interrompe**. Isso evita treinar com uma amostra incorreta.

In [ ]:
script=SCRIPTS/'executar_treinamento_real_area1.py'
subprocess.run([sys.executable,str(script),
    '--area1',str(AREA1),'--out',str(SAIDA),
    '--so-auditoria'],check=True)
import json
auditoria=json.loads((SAIDA/'auditoria_arquivos_area1.json').read_text(encoding='utf-8'))
assert auditoria['total_pontos_referencia']==510
print('Árvores:',auditoria['quantidades_referencia']['arvore'])
print('Gramíneas:',auditoria['quantidades_referencia']['grama'])
print('CRS:',auditoria['raster']['crs'])
print('Imagens com mesma grade?:',
      auditoria['raster']['grade_geotransformacao_identica'])

## 4. Executar a reconstrução, o treinamento e a validação

**Esta é a execução principal.** Não utiliza pontos de teste das outras regiões. Verifica rigorosamente que foram obtidos **505 segmentos rotulados únicos** e que os oito modelos foram realmente salvos.

Os arquivos de progresso permitem retomar a operação se o Colab desconectar. O comando **não altera** os arquivos originais.

In [ ]:
subprocess.run([sys.executable,str(script),
    '--area1',str(AREA1),'--out',str(SAIDA)],check=True)
relatorio=json.loads((SAIDA/'resumo_execucao_treinamento.json').read_text(encoding='utf-8'))
assert relatorio['estado']=='AREA1_TREINADA_E_AUDITADA'
assert relatorio['area1_segmentos_rotulados_unicos']==505
assert len(relatorio['modelos'])==8
assert relatorio['nova_validacao_aninhada_concluida']
print('Treinamento e validação Area1 concluídos e registrados no seu Drive.')

## 5. Conferir as métricas internas, os modelos e os artefatos

Os dados exibidos abaixo só aparecem **depois de execução bem-sucedida**. Não confundir os resultados dessa nova divisão com as tabelas históricas da dissertação.

In [ ]:
import pandas as pd
from IPython.display import display
base=SAIDA/'modelos_fixos_area1'
valid=SAIDA/'validacao_aninhada_area1'
print('Modelos selecionados e F1 da validação interna de regularização:')
display(pd.read_csv(base/'modelos_escolhidos_area1.csv')[
    ['modelo','receita','f1_validacao_interna']])
print('Nova validação espacial aninhada (cinco divisões por escala):')
display(pd.read_csv(valid/'area1_resumo_oito_modelos.csv').sort_values(
    ['bloco_m','f1_macro_media'],ascending=[True,False]))
arquivos_modelos=list(base.glob('*.joblib'))
assert len(arquivos_modelos)==8
print('Modelos efetivamente salvos:',len(arquivos_modelos))
print('Relatório verificável:',SAIDA/'resumo_execucao_treinamento.json')
print('Próxima etapa: confirmar NIR regional e aplicar modelos congelados.')

## Interpretação científica

Esta execução realiza treinamento e **validação interna** usando somente a Area1. O conjunto de parâmetros treinado com os **505 segmentos** poderá ser transferido aos quatro rasters externos. Uma vez que Sagrada Família se sobrepõe à Area1, os pontos originais **não** deverão ser usados como teste independente ali.

**Para obter F1-macro externo**, serão necessárias amostras independentes de árvores e gramíneas em cada região. Enquanto não existirem, apenas mapas de previsão podem ser relatados.